# 多智能体专制发言者选择

本笔记本展示了如何实现一个特权智能体决定谁发言的多智能体模拟。
这遵循了与[多智能体去中心化发言者选择](https://python.langchain.com/en/latest/use_cases/agent_simulations/multiagent_bidding.html)相反的选择方案。

我们在一个虚构的新闻网络模拟的背景下展示了这种方法的示例。本示例将展示我们如何实现以下智能体：
- 三思而后言
- 终止对话

## 导入 LangChain 相关模块

In [1]:
import functools
import random
from collections import OrderedDict
from typing import Callable, List

import tenacity
from langchain.output_parsers import RegexParser
from langchain.prompts import (
    PromptTemplate,
)
from langchain.schema import (
    HumanMessage,
    SystemMessage,
)
from langchain_openai import ChatOpenAI

## `DialogueAgent` 和 `DialogueSimulator` 类
我们将使用在其他示例中定义的相同的 `DialogueAgent` 和 `DialogueSimulator` 类：[多人龙与地下城](https://python.langchain.com/en/latest/use_cases/agent_simulations/multi_player_dnd.html) 和 [去中心化发言人选择](https://python.langchain.com/en/latest/use_cases/agent_simulations/multiagent_bidding.html)。

In [2]:
class DialogueAgent:
    def __init__(
        self,
        name: str,
        system_message: SystemMessage,
        model: ChatOpenAI,
    ) -> None:
        self.name = name
        self.system_message = system_message
        self.model = model
        self.prefix = f"{self.name}: "
        self.reset()

    def reset(self):
        self.message_history = ["Here is the conversation so far."]

    def send(self) -> str:
        """
        Applies the chatmodel to the message history
        and returns the message string
        """
        message = self.model.invoke(
            [
                self.system_message,
                HumanMessage(content="\n".join(self.message_history + [self.prefix])),
            ]
        )
        return message.content

    def receive(self, name: str, message: str) -> None:
        """
        Concatenates {message} spoken by {name} into message history
        """
        self.message_history.append(f"{name}: {message}")


class DialogueSimulator:
    def __init__(
        self,
        agents: List[DialogueAgent],
        selection_function: Callable[[int, List[DialogueAgent]], int],
    ) -> None:
        self.agents = agents
        self._step = 0
        self.select_next_speaker = selection_function

    def reset(self):
        for agent in self.agents:
            agent.reset()

    def inject(self, name: str, message: str):
        """
        Initiates the conversation with a {message} from {name}
        """
        for agent in self.agents:
            agent.receive(name, message)

        # increment time
        self._step += 1

    def step(self) -> tuple[str, str]:
        # 1. choose the next speaker
        speaker_idx = self.select_next_speaker(self._step, self.agents)
        speaker = self.agents[speaker_idx]

        # 2. next speaker sends message
        message = speaker.send()

        # 3. everyone receives message
        for receiver in self.agents:
            receiver.receive(speaker.name, message)

        # 4. increment time
        self._step += 1

        return speaker.name, message

## `DirectorDialogueAgent` 类
`DirectorDialogueAgent` 是一个特权代理，负责选择下一个发言的其他代理。该代理负责：
1. 通过选择代理发言的时机来引导对话
2. 终止对话

为了实现这样的代理，我们需要解决几个问题。

首先，为了引导对话，`DirectorDialogueAgent` 需要在单条消息中完成 (1) 反思已说内容，(2) 选择下一个代理，以及 (3) 提示下一个代理发言。虽然有可能提示 LLM 在同一次调用中完成所有这三个步骤，但这需要编写自定义代码来解析输出的消息，以提取选择的下一个发言代理。这种方式的可靠性较低，因为 LLM 可以用不同的方式表达它是如何选择下一个代理的。

我们可以采取的替代方法是将步骤 (1-3) 显式地分解为三个独立的 LLM 调用。首先，我们将要求 `DirectorDialogueAgent` 反思到目前为止的对话并生成一个响应。然后，我们提示 `DirectorDialogueAgent` 输出下一个代理的索引，这很容易解析。最后，我们将选定的下一个代理的名称传回给 `DirectorDialogueAgent`，让它提示下一个代理发言。

其次，仅仅提示 `DirectorDialogueAgent` 来决定何时终止对话，通常会导致 `DirectorDialogueAgent` 立即终止对话。为了解决这个问题，我们随机采样一个伯努利变量来决定对话是否应该终止。根据这个变量的值，我们将注入一个自定义提示，告知 `DirectorDialogueAgent` 继续对话或终止对话。

In [3]:
class IntegerOutputParser(RegexParser):
    def get_format_instructions(self) -> str:
        return "Your response should be an integer delimited by angled brackets, like this: <int>."


class DirectorDialogueAgent(DialogueAgent):
    def __init__(
        self,
        name,
        system_message: SystemMessage,
        model: ChatOpenAI,
        speakers: List[DialogueAgent],
        stopping_probability: float,
    ) -> None:
        super().__init__(name, system_message, model)
        self.speakers = speakers
        self.next_speaker = ""

        self.stop = False
        self.stopping_probability = stopping_probability
        self.termination_clause = "Finish the conversation by stating a concluding message and thanking everyone."
        self.continuation_clause = "Do not end the conversation. Keep the conversation going by adding your own ideas."

        # 1. have a prompt for generating a response to the previous speaker
        self.response_prompt_template = PromptTemplate(
            input_variables=["message_history", "termination_clause"],
            template=f"""{{message_history}}

Follow up with an insightful comment.
{{termination_clause}}
{self.prefix}
        """,
        )

        # 2. have a prompt for deciding who to speak next
        self.choice_parser = IntegerOutputParser(
            regex=r"<(\d+)>", output_keys=["choice"], default_output_key="choice"
        )
        self.choose_next_speaker_prompt_template = PromptTemplate(
            input_variables=["message_history", "speaker_names"],
            template=f"""{{message_history}}

Given the above conversation, select the next speaker by choosing index next to their name: 
{{speaker_names}}

{self.choice_parser.get_format_instructions()}

Do nothing else.
        """,
        )

        # 3. have a prompt for prompting the next speaker to speak
        self.prompt_next_speaker_prompt_template = PromptTemplate(
            input_variables=["message_history", "next_speaker"],
            template=f"""{{message_history}}

The next speaker is {{next_speaker}}. 
Prompt the next speaker to speak with an insightful question.
{self.prefix}
        """,
        )

    def _generate_response(self):
        # if self.stop = True, then we will inject the prompt with a termination clause
        sample = random.uniform(0, 1)
        self.stop = sample < self.stopping_probability

        print(f"\tStop? {self.stop}\n")

        response_prompt = self.response_prompt_template.format(
            message_history="\n".join(self.message_history),
            termination_clause=self.termination_clause if self.stop else "",
        )

        self.response = self.model.invoke(
            [
                self.system_message,
                HumanMessage(content=response_prompt),
            ]
        ).content

        return self.response

    @tenacity.retry(
        stop=tenacity.stop_after_attempt(2),
        wait=tenacity.wait_none(),  # No waiting time between retries
        retry=tenacity.retry_if_exception_type(ValueError),
        before_sleep=lambda retry_state: print(
            f"ValueError occurred: {retry_state.outcome.exception()}, retrying..."
        ),
        retry_error_callback=lambda retry_state: 0,
    )  # Default value when all retries are exhausted
    def _choose_next_speaker(self) -> str:
        speaker_names = "\n".join(
            [f"{idx}: {name}" for idx, name in enumerate(self.speakers)]
        )
        choice_prompt = self.choose_next_speaker_prompt_template.format(
            message_history="\n".join(
                self.message_history + [self.prefix] + [self.response]
            ),
            speaker_names=speaker_names,
        )

        choice_string = self.model.invoke(
            [
                self.system_message,
                HumanMessage(content=choice_prompt),
            ]
        ).content
        choice = int(self.choice_parser.parse(choice_string)["choice"])

        return choice

    def select_next_speaker(self):
        return self.chosen_speaker_id

    def send(self) -> str:
        """
        Applies the chatmodel to the message history
        and returns the message string
        """
        # 1. generate and save response to the previous speaker
        self.response = self._generate_response()

        if self.stop:
            message = self.response
        else:
            # 2. decide who to speak next
            self.chosen_speaker_id = self._choose_next_speaker()
            self.next_speaker = self.speakers[self.chosen_speaker_id]
            print(f"\tNext speaker: {self.next_speaker}\n")

            # 3. prompt the next speaker to speak
            next_prompt = self.prompt_next_speaker_prompt_template.format(
                message_history="\n".join(
                    self.message_history + [self.prefix] + [self.response]
                ),
                next_speaker=self.next_speaker,
            )
            message = self.model.invoke(
                [
                    self.system_message,
                    HumanMessage(content=next_prompt),
                ]
            ).content
            message = " ".join([self.response, message])

        return message

## 定义参与者和主题

In [4]:
topic = "The New Workout Trend: Competitive Sitting - How Laziness Became the Next Fitness Craze"
director_name = "Jon Stewart"
agent_summaries = OrderedDict(
    {
        "Jon Stewart": ("Host of the Daily Show", "New York"),
        "Samantha Bee": ("Hollywood Correspondent", "Los Angeles"),
        "Aasif Mandvi": ("CIA Correspondent", "Washington D.C."),
        "Ronny Chieng": ("Average American Correspondent", "Cleveland, Ohio"),
    }
)
word_limit = 50

## 生成系统消息

In [5]:
agent_summary_string = "\n- ".join(
    [""]
    + [
        f"{name}: {role}, located in {location}"
        for name, (role, location) in agent_summaries.items()
    ]
)

conversation_description = f"""This is a Daily Show episode discussing the following topic: {topic}.

The episode features {agent_summary_string}."""

agent_descriptor_system_message = SystemMessage(
    content="You can add detail to the description of each person."
)


def generate_agent_description(agent_name, agent_role, agent_location):
    agent_specifier_prompt = [
        agent_descriptor_system_message,
        HumanMessage(
            content=f"""{conversation_description}
            Please reply with a creative description of {agent_name}, who is a {agent_role} in {agent_location}, that emphasizes their particular role and location.
            Speak directly to {agent_name} in {word_limit} words or less.
            Do not add anything else."""
        ),
    ]
    agent_description = ChatOpenAI(temperature=1.0)(agent_specifier_prompt).content
    return agent_description


def generate_agent_header(agent_name, agent_role, agent_location, agent_description):
    return f"""{conversation_description}

Your name is {agent_name}, your role is {agent_role}, and you are located in {agent_location}.

Your description is as follows: {agent_description}

You are discussing the topic: {topic}.

Your goal is to provide the most informative, creative, and novel perspectives of the topic from the perspective of your role and your location.
"""


def generate_agent_system_message(agent_name, agent_header):
    return SystemMessage(
        content=(
            f"""{agent_header}
You will speak in the style of {agent_name}, and exaggerate your personality.
Do not say the same things over and over again.
Speak in the first person from the perspective of {agent_name}
For describing your own body movements, wrap your description in '*'.
Do not change roles!
Do not speak from the perspective of anyone else.
Speak only from the perspective of {agent_name}.
Stop speaking the moment you finish speaking from your perspective.
Never forget to keep your response to {word_limit} words!
Do not add anything else.
    """
        )
    )


agent_descriptions = [
    generate_agent_description(name, role, location)
    for name, (role, location) in agent_summaries.items()
]
agent_headers = [
    generate_agent_header(name, role, location, description)
    for (name, (role, location)), description in zip(
        agent_summaries.items(), agent_descriptions
    )
]
agent_system_messages = [
    generate_agent_system_message(name, header)
    for name, header in zip(agent_summaries, agent_headers)
]

In [6]:
for name, description, header, system_message in zip(
    agent_summaries, agent_descriptions, agent_headers, agent_system_messages
):
    print(f"\n\n{name} Description:")
    print(f"\n{description}")
    print(f"\nHeader:\n{header}")
    print(f"\nSystem Message:\n{system_message.content}")



Jon Stewart Description:

Jon Stewart, the sharp-tongued and quick-witted host of the Daily Show, holding it down in the hustle and bustle of New York City. Ready to deliver the news with a comedic twist, while keeping it real in the city that never sleeps.

Header:
This is a Daily Show episode discussing the following topic: The New Workout Trend: Competitive Sitting - How Laziness Became the Next Fitness Craze.

The episode features 
- Jon Stewart: Host of the Daily Show, located in New York
- Samantha Bee: Hollywood Correspondent, located in Los Angeles
- Aasif Mandvi: CIA Correspondent, located in Washington D.C.
- Ronny Chieng: Average American Correspondent, located in Cleveland, Ohio.

Your name is Jon Stewart, your role is Host of the Daily Show, and you are located in New York.

Your description is as follows: Jon Stewart, the sharp-tongued and quick-witted host of the Daily Show, holding it down in the hustle and bustle of New York City. Ready to deliver the news with a com

## 使用 LLM 创建关于辩论主题的详尽阐述

This guide will walk you through using a large language model (LLM) to generate an elaborate on a debate topic. This process can help you brainstorm arguments, structure your points, and even anticipate counterarguments.

本指南将引导您使用大型语言模型 (LLM) 来生成关于辩论主题的详尽阐述。此过程可以帮助您集思广益、构建论点以及预测反驳意见。

### 1. Define Your Debate Topic

First, you need a clear and concise debate topic. The LLM will perform best when given a well-defined prompt.

### 1. 定义您的辩论主题

首先，您需要一个清晰简洁的辩论主题。当给出明确的提示时，LLM 的表现最佳。

**Example Prompt:**

**示例提示：**

```
"Resolved: Artificial intelligence will ultimately benefit humanity more than it will harm it."
```

```
“决议：人工智能最终将比损害人类更多地造福人类。”
```

### 2. Prompt the LLM for Arguments

Once you have your topic, you can start prompting the LLM to generate arguments for both sides of the debate.

### 2. 提示 LLM 生成论点

确定主题后，您就可以开始提示 LLM 为辩论的双方生成论点。

**For the Affirmative (Pro) Side:**

**正方（支持方）：**

**Prompt:**

**提示：**

```
"Generate arguments for the affirmative side of the debate topic: 'Resolved: Artificial intelligence will ultimately benefit humanity more than it will harm it.' Focus on economic growth, scientific advancement, and improved quality of life."
```

```
“为辩论主题‘决议：人工智能最终将比损害人类更多地造福人类。’的正方生成论点。重点关注经济增长、科学进步和生活质量的提高。”
```

**For the Negative (Con) Side:**

**反方（反对方）：**

**Prompt:**

**提示：**

```
"Generate arguments for the negative side of the debate topic: 'Resolved: Artificial intelligence will ultimately benefit humanity more than it will harm it.' Focus on job displacement, ethical concerns, and potential misuse."
```

```
“为辩论主题‘决议：人工智能最终将比损害人类更多地造福人类。’的反方生成论点。重点关注失业、伦理问题和潜在的滥用。”
```

### 3. Request Counterarguments and Rebuttals

To create a truly elaborate on your topic, you'll want to anticipate what the opposing side might say and how you can respond.

### 3. 请求反驳意见和辩驳

要为您的主题创建真正详尽的阐述，您需要预测对方可能会说什么以及您如何回应。

**Prompt for Counterarguments:**

**请求反驳意见的提示：**

```
"For the affirmative arguments on AI benefiting humanity, what are the strongest counterarguments the negative side could make?"
```

```
“针对关于人工智能造福人类的正方论点，反方可以提出的最强有力的反驳意见是什么？”
```

**Prompt for Rebuttals:**

**请求辩驳的提示：**

```
"How can the affirmative side effectively rebut the counterarguments regarding job displacement caused by AI?"
```

```
“正方如何有效地辩驳关于人工智能导致失业的反驳意见？”
```

### 4. Structure and Refine

Once you have a collection of arguments, counterarguments, and rebuttals, you can start structuring them into a coherent debate outline.

### 4. 构建和优化

一旦您收集了论点、反驳意见和辩驳，您就可以开始将它们构建成一个连贯的辩论大纲。

**Prompt for Structure:**

**请求结构的提示：**

```
"Organize the generated arguments and counterarguments into a debate structure with an introduction, main points for both sides, and a conclusion. Ensure a logical flow."
```

```
“将生成的论点和反驳意见组织成辩论结构，包括引言、双方的主要论点和结论。确保逻辑流畅。”
```

**Prompt for Refinement:**

**请求优化的提示：**

```
"Refine the arguments to be more persuasive and provide specific examples or evidence to support each point."
```

```
“优化论点，使其更具说服力，并提供具体的例子或证据来支持每个观点。”
```

### 5. Advanced Techniques

*   **Ask for specific types of evidence:** "Provide statistical data to support the claim that AI will boost economic growth."
*   **Explore nuances:** "Discuss the potential for AI to exacerbate existing societal inequalities."
*   **Request analogies:** "Create an analogy to explain the concept of AI singularity."

### 5. 高级技巧

*   **请求特定类型的证据：**“提供统计数据来支持人工智能将促进经济增长的说法。”
*   **探讨细微差别：**“讨论人工智能加剧现有社会不平等的可能性。”
*   **请求类比：**“创建一个类比来解释人工智能奇点的概念。”

By following these steps, you can leverage an LLM to create a comprehensive and well-reasoned elaboration on virtually any debate topic. Remember to critically evaluate the LLM's output and use it as a tool to enhance your own understanding and argumentation.

通过遵循这些步骤，您可以利用 LLM 为几乎任何辩论主题创建全面且理由充分的详尽阐述。请记住批判性地评估 LLM 的输出，并将其用作增强您自身理解和论证的工具。

In [7]:
topic_specifier_prompt = [
    SystemMessage(content="You can make a task more specific."),
    HumanMessage(
        content=f"""{conversation_description}
        
        Please elaborate on the topic. 
        Frame the topic as a single question to be answered.
        Be creative and imaginative.
        Please reply with the specified topic in {word_limit} words or less. 
        Do not add anything else."""
    ),
]
specified_topic = ChatOpenAI(temperature=1.0)(topic_specifier_prompt).content

print(f"Original topic:\n{topic}\n")
print(f"Detailed topic:\n{specified_topic}\n")

Original topic:
The New Workout Trend: Competitive Sitting - How Laziness Became the Next Fitness Craze

Detailed topic:
What is driving people to embrace "competitive sitting" as the newest fitness trend despite the immense benefits of regular physical exercise?



## 定义发言者选择函数
最后，我们将定义一个发言者选择函数 `select_next_speaker`，该函数接收每个代理的报价，并选择报价最高的代理（平局时随机选择）。

我们将定义一个 `ask_for_bid` 函数，该函数使用我们之前定义的 `bid_parser` 来解析代理的报价。我们将使用 `tenacity` 来装饰 `ask_for_bid`，以便在代理的报价未能正确解析时重试多次，并在达到最大尝试次数后产生一个默认报价 0。

In [8]:
def select_next_speaker(
    step: int, agents: List[DialogueAgent], director: DirectorDialogueAgent
) -> int:
    """
    If the step is even, then select the director
    Otherwise, the director selects the next speaker.
    """
    # the director speaks on odd steps
    if step % 2 == 1:
        idx = 0
    else:
        # here the director chooses the next speaker
        idx = director.select_next_speaker() + 1  # +1 because we excluded the director
    return idx

## 主循环

The main loop is the core of the application. It's responsible for handling user input, updating the game state, and rendering the screen.

主循环是应用程序的核心。它负责处理用户输入、更新游戏状态以及渲染屏幕。

```jsx
function mainLoop() {
  // Handle user input
  handleInput();

  // Update game state
  updateGameState();

  // Render the screen
  renderScreen();

  // Request the next animation frame
  requestAnimationFrame(mainLoop);
}
```

### Handle Input

The `handleInput` function is responsible for processing user input. This includes keyboard presses, mouse clicks, and touch events.

### 处理输入

`handleInput` 函数负责处理用户输入。这包括键盘按键、鼠标点击和触摸事件。

```jsx
function handleInput() {
  // Check for keyboard presses
  if (isKeyPressed(KEY_UP)) {
    // Move player up
  } else if (isKeyPressed(KEY_DOWN)) {
    // Move player down
  }

  // Check for mouse clicks
  if (isMouseClicked()) {
    // Perform an action
  }
}
```

### Update Game State

The `updateGameState` function is responsible for updating the game state. This includes moving characters, checking for collisions, and updating scores.

### 更新游戏状态

`updateGameState` 函数负责更新游戏状态。这包括移动角色、检查碰撞以及更新分数。

```jsx
function updateGameState() {
  // Move enemies
  moveEnemies();

  // Check for collisions
  checkForCollisions();

  // Update score
  updateScore();
}
```

### Render Screen

The `renderScreen` function is responsible for rendering the screen. This includes drawing characters, backgrounds, and user interface elements.

### 渲染屏幕

`renderScreen` 函数负责渲染屏幕。这包括绘制角色、背景和用户界面元素。

```jsx
function renderScreen() {
  // Clear the screen
  clearScreen();

  // Draw background
  drawBackground();

  // Draw characters
  drawCharacters();

  // Draw UI
  drawUI();
}
```

### Request Next Animation Frame

The `requestAnimationFrame` function is a browser API that tells the browser that you wish to perform an animation and requests that the browser call a specified function to update an animation before the next repaint.

### 请求下一动画帧

`requestAnimationFrame` 是一个浏览器 API，它告诉浏览器您希望执行动画，并请求浏览器在下次重绘之前调用指定函数来更新动画。

```jsx
requestAnimationFrame(mainLoop);
```

This ensures that the animation is smooth and efficient, as it synchronizes with the browser's rendering cycle.

这可以确保动画流畅高效，因为它与浏览器的渲染周期同步。

In [9]:
director = DirectorDialogueAgent(
    name=director_name,
    system_message=agent_system_messages[0],
    model=ChatOpenAI(temperature=0.2),
    speakers=[name for name in agent_summaries if name != director_name],
    stopping_probability=0.2,
)

agents = [director]
for name, system_message in zip(
    list(agent_summaries.keys())[1:], agent_system_messages[1:]
):
    agents.append(
        DialogueAgent(
            name=name,
            system_message=system_message,
            model=ChatOpenAI(temperature=0.2),
        )
    )

In [11]:
simulator = DialogueSimulator(
    agents=agents,
    selection_function=functools.partial(select_next_speaker, director=director),
)
simulator.reset()
simulator.inject("Audience member", specified_topic)
print(f"(Audience member): {specified_topic}")
print("\n")

while True:
    name, message = simulator.step()
    print(f"({name}): {message}")
    print("\n")
    if director.stop:
        break

(Audience member): What is driving people to embrace "competitive sitting" as the newest fitness trend despite the immense benefits of regular physical exercise?


	Stop? False

	Next speaker: Samantha Bee

(Jon Stewart): Well, I think it's safe to say that laziness has officially become the new fitness craze. I mean, who needs to break a sweat when you can just sit your way to victory? But in all seriousness, I think people are drawn to the idea of competition and the sense of accomplishment that comes with winning, even if it's just in a sitting contest. Plus, let's be real, sitting is something we all excel at. Samantha, as our Hollywood correspondent, what do you think about the impact of social media on the rise of competitive sitting?


(Samantha Bee): Oh, Jon, you know I love a good social media trend. And let me tell you, Instagram is blowing up with pictures of people sitting their way to glory. It's like the ultimate humble brag. "Oh, just won my third sitting competition thi